# Priority Queue Pruning — Algorithm Exploration

This notebook walks through the **lazy greedy priority queue pruning** algorithm step by step on UTD19 data with a single gamma value.

## Algorithm overview

The algorithm has **two phases**:

### Phase 1: Lazy greedy forward selection
Instead of evaluating *every* remaining model at each iteration (like `ForwardSelectionPruner` does), we use a **max-heap** with **lazy re-evaluation**:

1. Start with $S = \emptyset$
2. Compute the **gain** $g(m|S) = E(S) - E(S \cup \{m\})$ for all models $m$ and push into a max-heap
3. Pop the top element (highest gain):
   - If its gain was computed for the **current** $S$ (fresh): accept if gain > 0, **or** if gain = 0 but $E(S) > \gamma$ (safe because adding a model can never increase $E(S)$)
   - If **stale** (computed for an older $S$): recompute gain, push back, try again
4. After each accepted add: if $E(S) \leq \gamma$, exit phase 1

**Key insight**: $E(S) = \max_i U(i|S)$ is *not* submodular, so a single model addition may not reduce the bottleneck (gain = 0). But adding it is never harmful, and it can enable future reductions. We only stop when $E(S) \leq \gamma$ or all models are exhausted.

### Phase 2: Simple backward cleanup
Sweep through $S$ and remove any model whose removal keeps $E(S) \leq \gamma$. Repeat until stable.

This catches redundancies that crept in during the greedy forward pass.

In [ ]:
import heapq
import os
import sys
import time

import numpy as np

# Ensure project root is on path
_project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

from epcrc.coverage import CoverageFunctional
from epcrc.pruning import PriorityQueuePruner
from epcrc.utd19_pipeline import build_or_load_bundle

GAMMA = 60.0

## Load the UTD19 bundle

Same cached bundle used by the other experiments.

In [ ]:
DATA_CSV = os.path.join(_project_root, "data", "utd19_u.csv")
CACHE_DIR = os.path.join(_project_root, "data", "exp_0_utd19_cache")

bundle = build_or_load_bundle(data_csv=DATA_CSV, cache_dir=CACHE_DIR, verbose=True)
model_names = bundle.model_names
N = len(model_names)

print(f"\nN = {N} models")
print(f"Y_fit  = {bundle.Y_fit.shape}")
print(f"Y_eval = {bundle.Y_eval.shape}")
print(f"Models: {model_names}")

## Step 1: Bootstrap — pick the best first model

$E(\emptyset) = \infty$, so gain-based ranking is meaningless for the first pick. Instead, we evaluate every singleton $\{m\}$ and pick the one with the **lowest** $E(\{m\})$.

After that, gains are computed relative to a finite $E(S)$.

In [ ]:
coverage_fn = CoverageFunctional(
    Y_fit=bundle.Y_fit,
    Y_eval=bundle.Y_eval,
    model_names=model_names,
    metric="mean_abs",
)

# Evaluate all singletons
singleton_coverage = []
for m in range(N):
    E_m, _ = coverage_fn.compute_coverage({m})
    singleton_coverage.append((m, E_m))

# Sort by E({m}) ascending — best first model at top
singleton_coverage.sort(key=lambda x: x[1])

print(f"{'Model':>25s}  {'E({m})':>10s}")
print(f"{'-'*25}  {'-'*10}")
for m, e in singleton_coverage:
    print(f"{model_names[m]:>25s}  {e:>10.4f}")

# Bootstrap: pick the best
best_first_m, best_first_E = singleton_coverage[0]
S = {best_first_m}
E_current = best_first_E
print(f"\nBootstrap: pick {model_names[best_first_m]} with E({{{model_names[best_first_m]}}}) = {E_current:.4f}")

## Step 2: Build the max-heap (gains relative to the bootstrap singleton)

Now that $S = \{m^*\}$ with finite $E(S)$, we compute gains $g(m|S) = E(S) - E(S \cup \{m\})$ for all remaining models and push into a max-heap.

Each entry is a tuple `(-gain, timestamp, model_idx, generation)`:
- **Negated gain**: so the smallest tuple = highest gain
- **Timestamp**: tie-breaker for FIFO ordering when gains are equal  
- **Generation**: tracks when this gain was computed. If `entry_gen < current_gen`, the entry is **stale** and needs recomputation

In [ ]:
generation = 0
timestamp = 0
heap = []

# Compute gains relative to bootstrap singleton S
remaining = set(range(N)) - S
for m in sorted(remaining):
    E_with_m, _ = coverage_fn.compute_coverage(S | {m})
    g = E_current - E_with_m
    heapq.heappush(heap, (-g, timestamp, m, generation))
    timestamp += 1

# Show the top of the heap (highest gains)
print(f"Gains computed relative to S = {{{model_names[best_first_m]}}}, E(S) = {E_current:.4f}\n")
print(f"  {'Rank':>4s}  {'Model':>25s}  {'gain':>10s}  {'gen':>4s}")
print(f"  {'----':>4s}  {'-'*25}  {'-'*10}  {'-'*4}")

heap_sorted = sorted(heap)
for rank, (neg_g, ts, m, gen) in enumerate(heap_sorted[:10], 1):
    print(f"  {rank:>4d}  {model_names[m]:>25s}  {-neg_g:>10.4f}  {gen:>4d}")
if len(heap_sorted) > 10:
    print(f"  ... ({len(heap_sorted) - 10} more)")

print(f"\n  Total heap size: {len(heap)}")

## Step 3: The pop → freshness check → recompute/accept loop

This is the heart of the algorithm. Watch how:
1. The **first pop** is always fresh (generation matches) → accepted immediately
2. After accepting, all other entries become **stale** (generation incremented)
3. Subsequent pops require **recomputation** before they can be accepted
4. Often a model gets recomputed and pushed back, only for the *same* model to float back to the top → accepted on second try

This is where the efficiency win happens: we only recompute gains for the few models that bubble to the top, not all $N$ remaining.

In [ ]:
# Manual walk-through: trace the first few accepted adds (after bootstrap)
E_trace = E_current  # starts from bootstrap singleton
S_trace = set(S)     # starts from bootstrap singleton
gen = 0
ts = timestamp
heap_trace = list(heap)  # copy
heapq.heapify(heap_trace)

max_accepts = 6  # show detailed trace for first few accepted adds
accepted = 0
total_pops = 0

while heap_trace and accepted < max_accepts:
    neg_g, _, m_idx, entry_gen = heapq.heappop(heap_trace)
    gain = -neg_g
    total_pops += 1

    if m_idx in S_trace:
        continue

    if entry_gen == gen:
        # FRESH entry
        if gain > 0 or (gain >= 0 and E_trace > GAMMA):
            accepted += 1
            S_trace.add(m_idx)
            E_trace, _ = coverage_fn.compute_coverage(S_trace)
            gen += 1
            print(f"--- Accept #{accepted} (after {total_pops} total pops) ---")
            print(f"  Model:  {model_names[m_idx]}")
            print(f"  Gain:   {gain:.4f}  (FRESH, gen={entry_gen})")
            print(f"  S now:  {{{', '.join(model_names[i] for i in sorted(S_trace))}}}")
            print(f"  E(S):   {E_trace:.4f}  |S|={len(S_trace)}")
            if E_trace <= GAMMA:
                print(f"  ** E(S) = {E_trace:.4f} <= gamma = {GAMMA} -> EXIT PHASE 1 **")
                break
            print()
        else:
            print(f"--- STOP: best fresh gain = {gain:.4f}, E(S)={E_trace:.4f} <= gamma ---")
            print(f"  Model: {model_names[m_idx]}")
            break
    else:
        # STALE entry — recompute
        E_with_m, _ = coverage_fn.compute_coverage(S_trace | {m_idx})
        new_gain = E_trace - E_with_m
        print(f"  [recompute] {model_names[m_idx]:>20s}: "
              f"old={gain:.4f} (gen {entry_gen}) -> new={new_gain:.4f} (gen {gen}), push back")
        heapq.heappush(heap_trace, (-new_gain, ts, m_idx, gen))
        ts += 1

print(f"\nTotal pops: {total_pops}, Accepted adds: {accepted}")

## Step 4: Full run with `PriorityQueuePruner`

Now run the actual class implementation (from `epcrc/pruning.py`) with `debug=True` to see both phases.

Phase 2 (backward cleanup) will try removing each model from the set built in phase 1 — if removing it keeps $E(S) \leq \gamma$, the model was redundant.

In [ ]:
# Fresh coverage_fn (clean cache)
coverage_fn2 = CoverageFunctional(
    Y_fit=bundle.Y_fit,
    Y_eval=bundle.Y_eval,
    model_names=model_names,
    metric="mean_abs",
)

pruner = PriorityQueuePruner(coverage_fn=coverage_fn2, tolerance_gamma=GAMMA)

t0 = time.time()
result = pruner.run(debug=True)
elapsed = time.time() - t0

print(f"\nWall time: {elapsed:.2f}s")

## Step 5: Final kept set and E(S)

In [ ]:
print(f"gamma = {GAMMA}")
print(f"|S|   = {len(result.kept_set)}")
print(f"E(S)  = {result.coverage:.4f}")
print(f"sum U = {result.sum_uniqueness:.4f}")
print(f"\nKept models:")
for i in sorted(result.kept_set):
    print(f"  {model_names[i]}")

print(f"\nRemoved models:")
for i in range(N):
    if i not in result.kept_set:
        print(f"  {model_names[i]}")

print(f"\nFull history:")
print(f"  {'iter':>4s} {'action':>8s} {'model':>25s} {'|S|':>4s} {'E(S)':>10s}")
print(f"  {'----':>4s} {'------':>8s} {'-'*25}  {'---':>4s} {'----':>10s}")
for step in result.history:
    name = step.removed_model_name or "(stop)"
    print(
        f"  {step.iteration:>4d} {step.action:>8s} {name:>25s} "
        f"{len(step.kept_set):>4d} {step.coverage:>10.4f}"
    )

## Quick comparison with forward selection

Run the standard (non-lazy) forward selection on the same gamma to see if the priority queue approach finds the same or a different solution.

In [ ]:
from epcrc.pruning import ForwardSelectionPruner, BackwardEliminationPruner

# Forward selection
cov_fwd = CoverageFunctional(Y_fit=bundle.Y_fit, Y_eval=bundle.Y_eval,
                              model_names=model_names, metric="mean_abs")
t0 = time.time()
result_fwd = ForwardSelectionPruner(cov_fwd, GAMMA).run()
t_fwd = time.time() - t0

# Backward elimination
cov_bwd = CoverageFunctional(Y_fit=bundle.Y_fit, Y_eval=bundle.Y_eval,
                              model_names=model_names, metric="mean_abs")
t0 = time.time()
result_bwd = BackwardEliminationPruner(cov_bwd, GAMMA).run()
t_bwd = time.time() - t0

print(f"{'Algorithm':>25s}  {'|S|':>4s}  {'E(S)':>10s}  {'sum U':>10s}  {'time':>8s}")
print(f"{'-'*25}  {'---':>4s}  {'-'*10}  {'-'*10}  {'-'*8}")
for name, res, t in [
    ("priority_queue", result, elapsed),
    ("forward_selection", result_fwd, t_fwd),
    ("backward_elimination", result_bwd, t_bwd),
]:
    print(f"{name:>25s}  {len(res.kept_set):>4d}  {res.coverage:>10.4f}  "
          f"{res.sum_uniqueness:>10.4f}  {t:>7.2f}s")

print(f"\nKept sets:")
print(f"  PQ:       {sorted(model_names[i] for i in result.kept_set)}")
print(f"  Forward:  {sorted(model_names[i] for i in result_fwd.kept_set)}")
print(f"  Backward: {sorted(model_names[i] for i in result_bwd.kept_set)}")